In [0]:
from pyspark.sql.functions import *

hub_customer = spark.table("default.hub_customer")
hub_order = spark.table("default.hub_order")
hub_product = spark.table("default.hub_product")

clean_customer = spark.table("default.clean_customer")
raw_orders = spark.table("default.raw_orders")
raw_products = spark.table("default.raw_products")

In [0]:
clean_customer.printSchema()
raw_orders.printSchema()
raw_products.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- OrderID: string (nullable = true)
 |-- CustID: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: long (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)



In [0]:
customer_sat = clean_customer.alias("c") \
.join(
    hub_customer.alias("h"),
    col("c.Customer_ID") == col("h.Customer_ID"),
    "inner"
)

In [0]:
sat_customer = customer_sat.select(

    col("h.Customer_HK").alias("Customer_HK"),

    col("c.Name").alias("Customer_Name"),

    col("c.Email"),

    col("c.City"),

    sha2(
        concat_ws(
            "|",
            col("c.Name"),
            col("c.Email"),
            col("c.City")
        ),
        256
    ).alias("HashDiff"),

    col("c.Load_Date").alias("Load_Date"),

    col("c.Record_Source").alias("Record_Source")

)

In [0]:
%sql
DROP TABLE IF EXISTS default.sat_customer;

In [0]:
sat_customer.write.mode("overwrite").saveAsTable("default.sat_customer")

In [0]:
order_sat = raw_orders.alias("o") \
.join(
    hub_order.alias("h"),
    col("o.OrderID") == col("h.OrderID"),
    "inner"
)

In [0]:
sat_order = order_sat.select(

    col("h.Order_HK").alias("Order_HK"),

    col("o.Amount"),

    col("o.OrderDate"),

    sha2(
        concat_ws(
            "|",
            col("o.Amount"),
            col("o.OrderDate")
        ),
        256
    ).alias("HashDiff"),

    col("o.Load_Date").alias("Load_Date"),

    col("o.Record_Source").alias("Record_Source")

)

In [0]:
%sql
DROP TABLE IF EXISTS default.sat_order;

In [0]:
sat_order.write.mode("overwrite").saveAsTable("default.sat_order")

In [0]:
product_sat = raw_products.alias("p") \
.join(
    hub_product.alias("h"),
    col("p.ProductID") == col("h.ProductID"),
    "inner"
)

In [0]:
raw_products.printSchema()

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: long (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)



In [0]:
sat_product = product_sat.select(

    col("h.Product_HK").alias("Product_HK"),

    col("p.ProductName").alias("Product_Name"),

    col("p.Category").alias("Category"),

    col("p.Price").alias("Price"),

    sha2(
        concat_ws(
            "|",
            col("p.ProductName"),
            col("p.Category"),
            col("p.Price")
        ),
        256
    ).alias("HashDiff"),

    col("p.Load_Date").alias("Load_Date"),

    col("p.Record_Source").alias("Record_Source")

)

In [0]:
%sql
DROP TABLE IF EXISTS default.sat_product;

In [0]:
sat_product.write.mode("overwrite").saveAsTable("default.sat_product")

In [0]:
%sql
SELECT COUNT(*) FROM default.sat_customer;

SELECT COUNT(*) FROM default.sat_order;

SELECT COUNT(*) FROM default.sat_product;

COUNT(*)
20


In [0]:
%sql
SELECT * FROM default.sat_customer LIMIT 10;

Customer_HK,Customer_Name,Email,City,HashDiff,Load_Date,Record_Source
a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07,Customer_1,customer1@gmail.com,City_2,8d7ef09c398a8fc4f2214c4ca3b47d8b5ba929fe0b21145c0ddb8634570357cd,2026-07-13T10:01:07.445Z,Customer_CSV
fdc98571fc93e8482954dcc8034bda27f571bd691cd86510a849dce99759933d,Customer_2,customer2@gmail.com,City_3,5682cc4efd8ffa1d19650189ea009de35fbc251a30732ce3f70d412f20d561b0,2026-07-13T10:01:07.445Z,Customer_CSV
64604487c74ef2fb0aca58ed3e951aacc1f537baf96c023b75441012e4b92880,Customer_3,customer3@gmail.com,City_4,add7d8c2cae5b60736a253725ef6c4c6caf7094d03344f6929bbdc3a73916c29,2026-07-13T10:01:07.445Z,Customer_CSV
f12ce50422ff0b232d51ad5764fad492b2f6bc3f1a74f3e5c04cc5d2742f9605,Customer_4,customer4@gmail.com,City_5,4c3f69d853c686b5fedeae7dc9819954cfd38eb267ce8d3e5f723983e03edb24,2026-07-13T10:01:07.445Z,Customer_CSV
95adbabc0d9bd895d140d98d75be9ad39f4d91cc7a53b37cfdfac32f5b75a332,Customer_5,customer5@gmail.com,City_6,0aa7ca5592698827074d663c0a58cc6fcb33b23827708dfc6f0944cbf4767f4f,2026-07-13T10:01:07.445Z,Customer_CSV
2db7a28cc4284c070736249ec3394b310cf755e62e74fc5693557c409618d9bc,Customer_6,customer6@gmail.com,City_7,36f30d2ec95ba3df7e3c525bc242290f57467b3404a649759bdd76d39156da18,2026-07-13T10:01:07.445Z,Customer_CSV
ac8d2a6d24e8a318136589c9cfd4684116f2ca05714ca7c377be40929db6de5a,Customer_7,customer7@gmail.com,City_8,adebb4f2206fc22c75c14c24e68cf4563b3c7d914f3abecfa866d9814d479910,2026-07-13T10:01:07.445Z,Customer_CSV
8d9e24cf98ed83b57da04302c28fb6d126f58019e590d2b8783b7ea38e546933,Customer_8,customer8@gmail.com,City_9,c68a4386969e864d73e6070597c07b76efb1bbf6ac57c6cbead30861760b6ea8,2026-07-13T10:01:07.445Z,Customer_CSV
de36046be14c2817aceb8feb35714eaf11449ff97e5e986cb01888b78989ae3b,Customer_9,customer9@gmail.com,City_10,7ff8b515de624a41b3ef66c9b7c309b83422cb794f8c372eaa9845402f5e7253,2026-07-13T10:01:07.445Z,Customer_CSV
e5f231360af6b3cc1ea64abb14e6549a75ca2d9fa6a531df7f9c3ad7633671fc,Customer_10,customer10@gmail.com,City_1,44a1e835ddfdf97eef8043df10f48211a98fb016dc49153de559bb081c1c6082,2026-07-13T10:01:07.445Z,Customer_CSV


In [0]:
%sql
SELECT * FROM default.sat_order LIMIT 10;

Order_HK,Amount,OrderDate,HashDiff,Load_Date,Record_Source
d70dd3111f66f42c31e54ce0535a53252b79bc1b383da336c6e0d31d5c2f32fe,617.0,2024-01-23,1eb33dcb5202e639d0ace916bc1925008a6ea957644e5995dfd203f6678e5d42,2026-07-13T10:07:07.112Z,Orders_CSV
378aeaa14440d732970510cea30074d508bdb8e91f7090e79779d494493bf5e4,687.5,2024-01-26,52d6a1d073962662e7989717a90e7db5827708614d1cf415e27fa03126af7816,2026-07-13T10:07:07.112Z,Orders_CSV
d3d075aab296fbc5adab56200b3c0f0fd00bef2d77ede38215e80b267a96e2db,1087.0,2024-01-15,059c0a49d447f944f8b31b992cfb1d09dcaab5b6ee478b0f4ef035908d233ff4,2026-07-13T10:07:07.112Z,Orders_CSV
a441382ff41e90c5951c1e51252f348801b14dbd36073cefcea86dfb82ea6275,170.5,2024-01-26,9c98ccae1ecc30e3aee0f7c49ded296d1f5fb1098cf26c43de7a707c12f28118,2026-07-13T10:07:07.112Z,Orders_CSV
47e505873fd1af927a99cfd9012c0859b404c40b23960f44c30d23c0e35e1fbe,241.0,2024-01-01,cc0a86d335aa120b5975aef2f0de6bb06f04798a2fb76d2c13de7e056dac2d79,2026-07-13T10:07:07.112Z,Orders_CSV
4c3c577e7b049c594b06d58bcf122908650c4b7cba75c80b01b68ceb6c31698b,335.0,2024-01-05,a7ab531f74e86e43a870472b816f535751e473f2e0634dbbaa82f7b1123c7ad4,2026-07-13T10:07:07.112Z,Orders_CSV
e705684af7f7073d60d1e04a1dfaee7ebc91786881391eb3f5ec309d4c38d1c1,452.5,2024-01-10,2578b77446600158c9bbb9ccda78fbfc79d1bffadff5254c4bae3e4e892a3dda,2026-07-13T10:07:07.112Z,Orders_CSV
768a62e4509a200a229d944627f37d66920c7217979df6c6ea7fb4db6d391b19,640.5,2024-01-18,b0be017f75cbccf10f151e87372a1076b2ef8bd275e5042cd640d9770944cc6c,2026-07-13T10:07:07.112Z,Orders_CSV
5cee17e5d7c51d53a4079a0d7f7394c6dbe63b92f71faa52c309b4d9d1b31e10,1016.5,2024-01-06,6af4e38d881687cc3684639f407a97a169fc5bbbcfd2b9c61424998364076a6b,2026-07-13T10:07:07.112Z,Orders_CSV
5f8b8578483b2875ac042f4591283571e446ed4495d786a74bcf663202ffecef,405.5,2024-01-02,a8f8f2d919c65cc59f1e2c16a9f5fd3bd3a7de66a45f4338da07091bc8403078,2026-07-13T10:07:07.112Z,Orders_CSV


In [0]:
%sql
SELECT * FROM default.sat_product LIMIT 10;

Product_HK,Product_Name,Category,Price,HashDiff,Load_Date,Record_Source
df1e40051eff4bcf9b4ebc93083bfcad7f5195746b3e657de6b72bf3cb8897c3,Product_1,Category_2,60,d65a9c2a974a46226e93d495c78575af6fd11ad3f63e56fd1b64651afaefdb32,2026-07-13T10:08:16.193Z,Products_CSV
eea502719605f8ec96582d94a0f1738190fe72d16c7b57de48fcfcebfbcb631a,Product_2,Category_3,70,e84a9a6f505305a3be743dfbc9bf3e6be1d847008d76efd13168fb45517184ae,2026-07-13T10:08:16.193Z,Products_CSV
12061cdd9a5c92ac2919a8567b74417f879ec93612941fb6bf4fdb941e7003eb,Product_3,Category_4,80,2329a9e8517f575a3f7ed7d2f4240e955873a88450f686731df8c82f44397c4c,2026-07-13T10:08:16.193Z,Products_CSV
00be3dd19f080c4255bba1ed6fcfbe7d8490f6a0299e336c906b212ee035bb80,Product_4,Category_5,90,a7f10f5fb94a389f107fe9848a0d0ef5dbb2349bf708129be468943e0ff7f0c4,2026-07-13T10:08:16.193Z,Products_CSV
e370b726b942a9efa03d66cc60370da2bcd962fb69ceba7f8e3d8d3a27d2a42f,Product_5,Category_1,100,f5bcd2521b850a479c6acbdccc80132f738a60b82b62c7cea7be87dbfc619cb9,2026-07-13T10:08:16.193Z,Products_CSV
7da069eca0948d1f28ee2d7757f32d086afc1866ea3879473d49af20aa515fbc,Product_6,Category_2,110,b77c166f6b35bd5e4d1519f96a6fc0a89583b290a6f5dfd2eff8b8b52eddae54,2026-07-13T10:08:16.193Z,Products_CSV
742fe1df5696bdb805ff8cfcf88538f236d7251f8dfea5a4f878a3ad09ee0e92,Product_7,Category_3,120,e6f86f8ffda7f1103a19673d046d04e6dbcb2159fae70debee0ef773c0a38723,2026-07-13T10:08:16.193Z,Products_CSV
e16c9047240f1c1c8816fa587c18b121486dd1969dc7f3c3a971b53fd7fc7235,Product_8,Category_4,130,efa99cf14a61834f8c085fba2371c6bc0960eb2c83d365cd31910e7161a1041b,2026-07-13T10:08:16.193Z,Products_CSV
28c093421a0d7dfbfad9457e923b75b9d08fd7784d4a52653bced05ade764a3f,Product_9,Category_5,140,da3f4712858b8167db3d6f1748cefed7dec7d8e086e3654b172103ed755837c6,2026-07-13T10:08:16.193Z,Products_CSV
229921285c7e2e77e64a07913b97f968aa4b54370f279a99cc81c8226f2fe54e,Product_10,Category_1,150,a7e7c4155478360730e32fcda6b0513dc4e5a94910ea385d2ae98a5e0aa84929,2026-07-13T10:08:16.193Z,Products_CSV
